In [65]:
import requests
import json
import pandas as pd
from dotenv import load_dotenv
import os

IGDB

Начнем с того, что попробуем взять igdb_id, igdb_secret, потом получить от них токен и попробовать с ним поработать для первичного анализа инди игр

Документация IGDB API- https://api-docs.igdb.com/ 



In [69]:
igdb_id = os.getenv('client_id')
igdb_secret = os.getenv('client_secret')


In [27]:
request = requests.post(url = f'https://id.twitch.tv/oauth2/token?client_id={igdb_id}&client_secret={igdb_secret}&grant_type=client_credentials')
request #получили токен, добавили в .env как access_token

<Response [200]>

In [70]:
access_token = os.getenv('access_token')


Имея датасет из 50к игр жанра Инди (имея их steam_id), получим их же id на igdb

Для начала попробуем получить 500 строк, если все будет в порядке, напишем цикл, который возьмет остальные нужные (так как в igdb можно брать максимум 500 строк за раз)

In [187]:
indie = pd.read_excel('indie_games.xlsx')
indie.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 52781 entries, 0 to 52780
Data columns (total 2 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   steam_app_id  52781 non-null  int64 
 1   name          52771 non-null  object
dtypes: int64(1), object(1)
memory usage: 824.8+ KB


In [188]:
ids = indie["steam_app_id"].astype(str).tolist()[:500]
uids = '","'.join(ids)
uids = '"' + uids + '"'


data = f'fields uid,game; where external_game_source = 1 & uid = ({uids}); limit 500;'
request_games_steam = requests.post('https://api.igdb.com/v4/external_games', headers={'Client-ID': igdb_id, 'Authorization': f'Bearer {access_token}'}, data=data)
res = request_games_steam.json()
res

map_df = pd.DataFrame(res)
map_df = map_df[["uid", "game"]].drop_duplicates(subset=["uid"])
map_df['uid'] = map_df['uid'].astype(int)

indie = indie.rename(columns ={'steam_app_id': 'uid'})
indie['uid'] = indie['uid'].astype(int)
out = indie.merge(map_df, on= 'uid', how= 'left')

In [189]:
out

,uid,name,game
0,578080,PUBG: BATTLEGROUNDS,27789.0
1,1623730,Palworld,151665.0
2,304930,Unturned,7878.0
3,105600,Terraria,1879.0
4,431960,Wallpaper Engine,NaN
...,...,...,...
52776,3591890,CombatBox,NaN
52777,3451960,101 Cats in New Delhi,NaN
52778,2656520,Mycelium Heaven,NaN
52779,1589920,Long Ago: A Puzzle Tale,NaN


In [ ]:
indie = indie.head(11_000).copy()
all = []

for i in range(0, len(indie), 500):
    ids = indie['uid'].iloc[i:i+500].astype(str).tolist()
    uids = '"' + '","'.join(ids) + '"'

    data = f'fields uid,game; where external_game_source = 1 & uid = ({uids}); limit 500;'
    request_games_steam = requests.post('https://api.igdb.com/v4/external_games', headers={'Client-ID': igdb_id, 'Authorization': f'Bearer {access_token}'}, data=data)
    res = request_games_steam.json()

    all += res
map_df = pd.DataFrame(all)
map_df = map_df[["uid", "game"]].drop_duplicates("uid")
map_df["uid"] = map_df["uid"].astype(int)

out = indie.merge(map_df, on="uid", how="left")
out

,uid,name,game
0,578080,PUBG: BATTLEGROUNDS,27789.0
1,1623730,Palworld,151665.0
2,304930,Unturned,7878.0
3,105600,Terraria,1879.0
4,431960,Wallpaper Engine,NaN
...,...,...,...
10995,316840,The Sacred Tears TRUE,9483.0
10996,796440,Spaceguy,86438.0
10997,775960,The Tower of Beatrice,90417.0
10998,726020,Mission Ammunition,74371.0


Что мы получили? 
1) uid - id игры в стиме
2) name - название игры
3) game - id игры в igdb

In [196]:
out = out.dropna() #удалим пропуски, так как далее не сможем с ними ничего сделать в других эндпоинтах (пропуски в game)
print(out.isna().sum())
out.info()

uid     0
name    0
game    0
dtype: int64
<class 'pandas.core.frame.DataFrame'>
Index: 10781 entries, 0 to 10999
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   uid     10781 non-null  int64  
 1   name    10781 non-null  object 
 2   game    10781 non-null  float64
dtypes: float64(1), int64(1), object(1)
memory usage: 336.9+ KB


In [212]:
out['game'] = out['game'].astype(int)
out.to_csv('igdb_steam_games.csv') 
out['game']

0         27789
1        151665
2          7878
3          1879
5         10233
          ...  
10995      9483
10996     86438
10997     90417
10998     74371
10999     98330
Name: game, Length: 10781, dtype: int64

In [219]:
prob = out.head(500).copy()
prob
ids = prob['game'].astype(str).tolist()
ids_str = ','.join(ids)

data1 = f'''
fields id, name, rating, rating_count, total_rating, total_rating_count, first_release_date,
language_supports, summary; where id = ({ids_str}); limit 500;
'''
request_prob = requests.post('https://api.igdb.com/v4/games', headers={'Client-ID': igdb_id, 'Authorization': f'Bearer {access_token}'}, data=data1)

res =request_prob.json()
res

[{'id': 111,
  'first_release_date': 1283904000,
  'name': 'Amnesia: The Dark Descent',
  'rating': 77.95333920297041,
  'rating_count': 523,
  'summary': 'Amnesia: The Dark Descent is a survival horror video game by Frictional Games. The game features a protagonist named Daniel exploring a dark and foreboding castle, while trying to maintain his sanity by avoiding monsters and other terrifying obstructions. The game was critically well received.',
  'total_rating': 83.47666960148521,
  'total_rating_count': 525,
  'language_supports': [478453,
   478456,
   478457,
   478454,
   478455,
   478458,
   478459,
   478460,
   478461,
   536120,
   536122,
   536123,
   536124,
   536125,
   536126,
   536127]},
 {'id': 232,
  'first_release_date': 942192000,
  'name': 'Half-Life: Opposing Force',
  'rating': 79.91431228594388,
  'rating_count': 386,
  'summary': 'Opposing Force returns to the same setting as Half-Life, but instead portrays the events from the perspective of a U.S. Marine,

In [220]:
df_games_igdb = pd.DataFrame(res)
df_games_igdb.isna().sum()

id                     0
first_release_date     0
name                   0
rating                15
rating_count          15
summary                0
total_rating          15
total_rating_count    15
language_supports      7
dtype: int64

In [221]:
df_games_igdb

,id,first_release_date,name,rating,rating_count,summary,total_rating,total_rating_count,language_supports
0,111,1283904000,Amnesia: The Dark Descent,77.953339,523.0,Amnesia: The Dark Descent is a survival horror...,83.476670,525.0,"[478453, 478456, 478457, 478454, 478455, 47845..."
1,232,942192000,Half-Life: Opposing Force,79.914312,386.0,Opposing Force returns to the same setting as ...,74.957156,387.0,"[604211, 604212, 604213, 604214]"
2,527,1321920000,Serious Sam 3: BFE,69.983462,172.0,Serious Sam 3: BFE is a first-person action sh...,70.408398,178.0,"[409265, 409267, 409269, 409272, 409273, 40927..."
3,576,1294704000,DC Universe Online,63.259002,65.0,DC Universe Online is a free-to-play superhero...,68.486644,72.0,"[137259, 137260, 137261, 137262, 137263, 75826..."
4,787,1128988800,Serious Sam II,71.570333,182.0,The iconic Serious Sam brings his trademark re...,74.951833,185.0,"[534874, 534875, 534873]"
...,...,...,...,...,...,...,...,...,...
494,281275,1703721600,Buckshot Roulette,77.749449,138.0,Buckshot Roulette is a tabletop horror game th...,78.874724,139.0,"[873748, 873749, 909410, 909411, 909412, 90941..."
495,309862,1774915200,TCG Card Shop Simulator,78.356938,22.0,Open your own local game store. Stock shelves ...,78.356938,22.0,"[1045833, 1063105, 1063106, 1063107, 1063108, ..."
496,311540,1723161600,Supermarket Together,84.839384,30.0,The multiplayer supermarket has arrived to you...,84.839384,30.0,"[1098181, 1098182, 1057452, 1057453, 1057454, ..."
497,317695,1738022400,Liar's Bar,69.880518,26.0,Liar's Bar is a 2-4 player multiplayer decepti...,69.880518,26.0,"[1093114, 1093118, 1093116, 1093117, 1093115, ..."


Пока что особо проблем с данными нет, есть несколько столбцов, которых нужно немного преобразовать (можно сделать, когда соберем полный датасет), но в общем и целом, с такими данными уже можно работать